# Description

```markdown
Problem      : Forcast
Resolve      : Learning : Deep Learning
Field        : Time series
Algorithm    : Neural Network
Network      : RNN
Architecture : 
Action       : 
Loss         : 
Train        : Supervised
Input        : Feature | Label
Output       : Forcast
Dataset      : Structured : Time series : jena_climate_2009_2016
```

# General

Import

In [ ]:
import os
import numpy as np
import pandas as pd
import cv2 as cv
from matplotlib import pyplot as plt
import keras as ks
import tensorflow as tf


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from keras import optimizers
from keras.utils import to_categorical
from keras import models, layers
from keras.models import load_model
from mtcnn import MTCNN
from keras.optimizers.schedules import ExponentialDecay

Variable

In [ ]:
DATA_DIR = '/Volumes/data/develop/ai'
PATH_DATASET = os.path.sep.join([DATA_DIR, "dataset", "jena_climate_2009_2016.csv"])
EPOCHS = 64
BATCH_SIZE = 32

# Dataset

Read

In [ ]:
dataset = pd.read_csv(PATH_DATASET)

Data

In [ ]:
data = dataset.iloc[:, 1:]

In [ ]:
print("Shape :", data.shape)
print("Type  :", type(data))
print("Data  :", data.iloc[0])

Label

In [ ]:
label = dataset.iloc[:, 2]

In [ ]:
print("Shape :", label.shape)
print("Type  :", type(label))
print("Label :", label.iloc[0])

Chart

In [ ]:
plt.style.use('ggplot')
plt.figure(figsize=(10, 5)) 
plt.plot(range(len(label)), label)
plt.title("Step and Label")
plt.xlabel("Step")
plt.ylabel("Label")
plt.show()

In [ ]:
plt.style.use('ggplot')
plt.figure(figsize=(10, 5)) 
plt.plot(range(1440), label[:1440])
plt.title("Step and Label")
plt.xlabel("Step")
plt.ylabel("Label")
plt.show()

Split data

In [ ]:
num_train_samples = int(0.5 * len(data))
num_val_samples = int(0.25 * len(data))
num_test_samples = len(data) - num_train_samples - num_val_samples

print("num_train_samples:", num_train_samples)
print("num_val_samples:", num_val_samples)
print("num_test_samples:", num_test_samples)

Normalize

In [ ]:
mean = data[:num_train_samples].mean(axis=0)
data -= mean
std = data[:num_train_samples].std(axis=0)
data /= std

Channel dimension 

In [ ]:
x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

Shape

In [ ]:
print("x_train : {}".format(x_train.shape))
print("x_test  : {}".format(x_test.shape))
print("y_train : {}".format(y_train.shape))
print("y_test  : {}".format(y_test.shape))

Label

In [ ]:
print("Label  : {}".format(y_train[2]))

Data

In [ ]:
plt.imshow(x_train[2])
plt.show()

# Create Pair

In [ ]:
def make_pairs(images, labels):
    
    pair_images = []
    pair_labels = []
    label_unique_count = len(np.unique(labels))
    label_index_same = [np.where(labels == i)[0] for i in range(0, label_unique_count)]

    for i in range(len(images)):
        image = images[i]
        label = labels[i]
        #---Add positive
        label_index_positive = np.random.choice(label_index_same[label])
        image_positive = images[label_index_positive]
        pair_images.append([image, image_positive])
        pair_labels.append([1])
        #---Add negative
        label_index_negative = np.where(labels != label)[0]
        image_negative = images[np.random.choice(label_index_negative)]
        pair_images.append([image, image_negative])
        pair_labels.append([0])
    
    return np.array(pair_images), np.array(pair_labels)

In [ ]:
(pair_train, label_train) = make_pairs(x_train, y_train)
(pair_test, label_test) = make_pairs(x_test, y_test)

In [ ]:
print("pair_train : {}".format(len(pair_train)))
print("pair_test  : {}".format(len(pair_test)))
print("label_train : {}".format(len(label_train)))
print("label_test  : {}".format(len(label_test)))

# Model

contrastive_loss

In [ ]:
def contrastive_loss(y, preds, margin=1):
	y = tf.cast(y, preds.dtype)
	squaredPreds = K.square(preds)
	squaredMargin = K.square(K.maximum(margin - preds, 0))
	loss = K.mean(y * squaredPreds + (1 - y) * squaredMargin)
	return loss

build_siamese_model

In [ ]:
def build_siamese_model(inputShape, embeddingDim=48):

    inputs = ks.layers.Input(inputShape)

    x = ks.layers.Conv2D(64, (2, 2), padding="same", activation="relu")(inputs)
    x = ks.layers.MaxPooling2D(pool_size=(2, 2))(x)
    x = ks.layers.Dropout(0.3)(x)

    x = ks.layers.Conv2D(64, (2, 2), padding="same", activation="relu")(x)
    x = ks.layers.MaxPooling2D(pool_size=2)(x)
    x = ks.layers.Dropout(0.3)(x)

    pooledOutput = ks.layers.GlobalAveragePooling2D()(x)

    outputs = ks.layers.Dense(embeddingDim)(pooledOutput)

    model = ks.Model(inputs, outputs)

    return model

euclidean_distance

In [ ]:
def euclidean_distance(vectors):
    (featsA, featsB) = vectors
    sumSquared = K.sum(K.square(featsA - featsB), axis=1, keepdims=True)
    return K.sqrt(K.maximum(sumSquared, K.epsilon()))

In [ ]:
featureExtractor = build_siamese_model(IMG_SHAPE)

imgA = ks.layers.Input(shape=IMG_SHAPE)
imgB = ks.layers.Input(shape=IMG_SHAPE)

featsA = featureExtractor(imgA)
featsB = featureExtractor(imgB)

distance = ks.layers.Lambda(euclidean_distance)([featsA, featsB])

#outputs = ks.layers.Dense(1, activation="sigmoid")(distance)

#mdl = ks.Model(inputs=[imgA, imgB], outputs=outputs)
#mdl.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

mdl = ks.Model(inputs=[imgA, imgB], outputs=distance)
mdl.compile(loss=contrastive_loss, optimizer="adam")

# Train

In [ ]:
h = mdl.fit(
    [pair_train[:, 0], pair_train[:, 1]], 
    label_train[:],
    validation_data=([pair_test[:, 0], pair_test[:, 1]], label_test[:]),
    batch_size=BATCH_SIZE, epochs=EPOCHS
)

# Evaluation

### Loss and Accuracy

In [ ]:
loss, accuracy = mdl.evaluate(x_test, y_test)
print("Loss: {:.2f} | Accuracy: {:.2f}".format(loss, accuracy))

### History

loss

In [ ]:
plt.plot(h.history["loss"], label="train_loss")
plt.plot(h.history["val_loss"], label="val_loss")
plt.legend(loc="lower left")
plt.xlabel("Epoch #")
plt.ylabel("Loss/Accuracy")
plt.title("Training Loss and Accuracy")
plt.style.use('ggplot')
plt.show()

accuracy

In [ ]:
plt.plot(h.history["accuracy"], label="train_acc")
plt.plot(h.history["val_accuracy"], label="val_acc")
plt.legend(loc="lower left")
plt.xlabel("Epoch #")
plt.ylabel("Loss/Accuracy")
plt.title("Training Loss and Accuracy")
plt.style.use('ggplot')
plt.show()

# Save model

In [ ]:
mdl.save(f"{data_dir}/model/siamese_1.h5")

# Test

In [ ]:
mdl = load_model(f"{data_dir}/model/siamese_1.h5")
for address in glob.glob(f"{dataset_predict_dir}/*.*"):
    #---Path
    path = os.path.dirname(address)
    name = os.path.splitext(os.path.basename(address))[0]
    extension = os.path.splitext(os.path.basename(address))[1] 
    #---Read image
    img = cv.imread(address)
    img = cv.cvtColor(img, cv.COLOR_BGR2RGB)
    #---Shape
    print("Shape: {}".format(img.shape))
    #---Detect Face
    img_detect = detector.detect_faces(img)
    if len(img_detect)>0:
        img_detect = img_detect[0]
        x, y, w, h = img_detect["box"]
        confidence = img_detect["confidence"]
        keypoints = img_detect["keypoints"]
        img = img[y:y+h, x:x+w]
    #---Preprocessing
    if len(img_detect)>0:
        img = cv.resize(img, (32, 32))
        img = img/255.0
        img = np.array([img])
    #---Predict
    if len(img_detect)>0:
        pred = mdl.predict(img)[0]
        pred_max = np.argmax(pred)
        out = output_label[pred_max]
    #---Display
    if len(img_detect)>0:    
        img = cv.imread(address)
        img = cv.cvtColor(img, cv.COLOR_BGR2RGB)
        confidence = "{:.2f}".format(confidence*100)

        cv.rectangle(img, (x, y), (x+w, y+h), colors[pred_max], 3)
        cv.putText(img, confidence, (x, y+60), cv.FONT_HERSHEY_PLAIN, 2, colors[pred_max], 2)
        cv.putText(img, out, (x, y+30), cv.FONT_HERSHEY_PLAIN, 2, colors[pred_max], 2)

        plt.imshow(img)
        plt.show()